# 路径参数和数值校验

与使用 `Query` 为查询参数声明更多的校验和元数据的方式相同，你也可以使用 Path 为路径参数声明相同类型的校验和元数据。

## 导入 Path
首先，从 `fastapi` 导入 `Path`：

In [ ]:
from typing import Annotated

from fastapi import FastAPI, Path, Query

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(
    item_id: Annotated[int, Path(title="The ID of the item to get")],
    q: Annotated[str | None, Query(alias="item-query")] = None,
):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    return results

## 声明元数据

你可以声明与 `Query` 相同的所有参数。

例如，要声明路径参数 `item_id`的 `title` 元数据值，你可以输入：

In [ ]:
from typing import Annotated

from fastapi import FastAPI, Path, Query

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(
    item_id: Annotated[int, Path(title="The ID of the item to get")],
    q: Annotated[str | None, Query(alias="item-query")] = None,
):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    return results

## 按需对参数排序¶
假设你想要声明一个必需的 str 类型查询参数 q。

而且你不需要为该参数声明任何其他内容，所以实际上你并不需要使用 Query。

但是你仍然需要使用 Path 来声明路径参数 item_id。

如果你将带有「默认值」的参数放在没有「默认值」的参数之前，Python 将会报错。

但是你可以对其重新排序，并将不带默认值的值（查询参数 q）放到最前面。

对 FastAPI 来说这无关紧要。它将通过参数的名称、类型和默认值声明（Query、Path 等）来检测参数，而不在乎参数的顺序。

因此，你可以将函数声明为：

In [ ]:
from fastapi import FastAPI, Path

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(q: str, 
                     item_id: int = Path(title="The ID of the item to get")):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    return results

## 按需对参数排序的技巧

如果你想不使用 `Query` 声明没有默认值的查询参数 `q`，同时使用 `Path` 声明路径参数 `item_id`，并使它们的顺序与上面不同，`Python` 对此有一些特殊的语法。

传递 * 作为函数的第一个参数。

Python 不会对该 * 做任何事情，但是它将知道之后的所有参数都应作为关键字参数（键值对），也被称为 kwargs，来调用。即使它们没有默认值。

In [ ]:
from fastapi import FastAPI, Path

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(*, 
                     item_id: int = Path(title="The ID of the item to get"), 
                     q: str):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    return results

## 数值校验：大于等于¶
使用 Query 和 Path（以及你将在后面看到的其他类）可以声明字符串约束，但也可以声明数值约束。

像下面这样，添加 ge=1 后，item_id 将必须是一个大于（greater than）或等于（equal）1 的整数。

In [ ]:
from fastapi import FastAPI, Path

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(
    *, item_id: int = Path(title="The ID of the item to get", ge=1), q: str
):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    return results

## 数值校验：大于和小于等于¶
同样的规则适用于：

- `gt`：大于（greater than）
- `le`：小于等于（less than or equal）

In [ ]:
from fastapi import FastAPI, Path

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(
    *,
    item_id: int = Path(title="The ID of the item to get", gt=0, le=1000),
    q: str,
):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    return results

## 数值校验：浮点数、大于和小于

数值校验同样适用于 `float` 值。

能够声明 `gt` 而不仅仅是 `ge` 在这个前提下变得重要起来。例如，你可以要求一个值必须大于 0，即使它小于 1。

因此，0.5 将是有效值。但是 0.0或 0 不是。

对于 `lt` 也是一样的。

In [ ]:
from fastapi import FastAPI, Path, Query

app = FastAPI()


@app.get("/items/{item_id}")
async def read_items(
    *,
    item_id: int = Path(title="The ID of the item to get", ge=0, le=1000),
    q: str,
    size: float = Query(gt=0, lt=10.5),
):
    results = {"item_id": item_id}
    if q:
        results.update({"q": q})
    if size:
        results.update({"size": size})
    return results

## 总结
你能够以与 查询参数和字符串校验 相同的方式使用 Query、Path（以及其他你还没见过的类）声明元数据和字符串校验。

而且你还可以声明数值校验：

- `gt`：大于（greater than）
- `ge`：大于等于（greater than or equal）
- `lt`：小于（less than）
- `le`：小于等于（less than or equal）